# Prompt Engineering & AI Productivity Lab
Techniques covered: zero-shot, few-shot, roles, context, constraints, structured JSON, summarization, classification, generation, AI-assisted debugging, prompt evaluation, and the reusable toolkit.

**Setup:** add `ANTHROPIC_API_KEY` in Colab (sidebar 🔑 Secrets, enable notebook access). Model names change over time; update `MODEL` if needed.

## 0. Setup

In [ ]:
!pip -q install anthropic
import os, json, re
from google.colab import userdata
os.environ["ANTHROPIC_API_KEY"] = userdata.get("ANTHROPIC_API_KEY")

import anthropic
client = anthropic.Anthropic()
MODEL = "claude-sonnet-5-5"

def ask(prompt, system="You are a helpful assistant.", temperature=0.2, max_tokens=700):
    r = client.messages.create(model=MODEL, max_tokens=max_tokens, temperature=temperature,
        system=system, messages=[{"role": "user", "content": prompt}])
    return "".join(b.text for b in r.content if b.type == "text").strip()

def show(title, text): print(f"\n=== {title} ===\n{text}")

## 1. Zero-shot vs few-shot

In [ ]:
msg = "Honestly the new dashboard is fine, I guess, but exporting takes forever."
zero = f'Classify the sentiment of this review as positive, negative, or mixed. Reply with one word.\n\nReview: {msg}'
few = f'''Classify the sentiment as positive, negative, or mixed. Reply with one word.

Review: Love it, works perfectly. -> positive
Review: Crashes constantly and support ignored me. -> negative
Review: Great design but way too expensive. -> mixed

Review: {msg} ->'''
show("Zero-shot", ask(zero, temperature=0))
show("Few-shot", ask(few, temperature=0))

**Notes:** few-shot examples anchor the label set and handle the ambiguous 'mixed' case more consistently.

## 2. Roles, context, instructions, constraints

In [ ]:
text = "Our onboarding takes 3 weeks because access requests wait on manual approvals from 4 separate teams."
prompts = {
 "Vague": f"Improve this: {text}",
 "Role only": f"You are a management consultant. Improve this: {text}",
 "Role + context + constraints": f'''You are a management consultant.
Context: I'm presenting to a COO who cares about cost and speed.
Task: Turn the problem statement below into a recommendation.
Constraints: max 60 words, one concrete first step, no jargon.

Problem: {text}''',
}
for k, p in prompts.items(): show(k, ask(p, temperature=0.4))

## 3. Structured outputs (JSON) with validation

In [ ]:
def extract_json(t):
    m = re.search(r"```(?:json)?\s*(.*?)```", t, re.S)
    if m: t = m.group(1)
    try: return json.loads(t)
    except json.JSONDecodeError:
        s, e = t.find("{"), t.rfind("}")
        return json.loads(t[s:e+1])

article = "On 3 March 2026, Priya Nair of Acme Corp signed a $250,000 contract with Globex to build a logistics platform."
p = f'''Extract fields. Use empty lists if not stated. Return ONLY JSON with keys:
"people", "organizations", "dates", "amounts" (all lists of strings), "summary" (max 15 words).

Text: {article}'''
data = extract_json(ask(p, temperature=0))
print(json.dumps(data, indent=2))
assert {"people","organizations","dates","amounts","summary"} <= set(data)

## 4. Summarization, classification, generation

In [ ]:
doc = '''The quarterly review showed revenue up 12% year over year, driven by the new subscription tier.
Churn rose from 3.1% to 3.6%, concentrated in monthly-plan customers. Billing support tickets doubled after
the pricing change. The team proposes annual-plan discounts and a billing FAQ before the next release.'''

show("Summary (constrained)", ask(f'''Summarize for an executive.
Exactly 3 bullets, each under 20 words, using only facts in the text.
End with a line "Action: ..." .

Text: {doc}'''))

tickets = ["I was billed $49 but my plan costs $29.", "Dashboard is blank after login.",
           "Change the email on my profile.", "Do you sell gift cards?"]
for t in tickets:
    r = ask(f'''Classify into exactly one of: billing, technical, account, other.
Return ONLY JSON: {{"category": "...", "confidence": 0-1}}\nMessage: {t}''', temperature=0)
    print(t, "->", extract_json(r))

show("Generation: email", ask('''Write a follow-up email. Context: demo went well yesterday. Recipient: Sam, client.
Goal: book a 30-minute call next week. Tone: warm, concise. Under 90 words.
Start with "Subject:". End with one clear call to action. No filler openers.''', temperature=0.5))

## 5. AI-assisted debugging

In [ ]:
buggy = '''def average(nums):
    total = 0
    for n in nums:
        total += n
    return total / len(nums)

print(average([]))'''
err = "ZeroDivisionError: division by zero"
show("Debug", ask(f'''Debug this Python code.
Return: 1) root cause (2 sentences) 2) fixed code 3) why it works 4) one pytest test.

Code:
```python
{buggy}
```
Error: {err}'''))

**Always run the suggested fix and test yourself** before trusting it.

## 6. Prompt evaluation: zero-shot vs few-shot

In [ ]:
CASES = [("I was billed $49 but my plan costs $29.","billing"),
 ("The dashboard shows a blank page after login.","technical"),
 ("Please change the email address on my profile.","account"),
 ("Where can I read your privacy policy?","other"),
 ("My card was declined even though it has funds.","billing"),
 ("API returns 500 when I send a JSON body.","technical"),
 ("I want to delete my account permanently.","account"),
 ("Can you recommend a good project management book?","other"),
 ("Refund me, this is the third time you double charged me!!","billing"),
 ("app not working","technical")]

BASE = 'Classify into exactly one of: billing, technical, account, other.\nReturn ONLY JSON: {"category": "..."}\n'
SHOTS = '''Examples:
Message: I was charged twice. -> {"category": "billing"}
Message: App crashes on upload. -> {"category": "technical"}
Message: Forgot which email I signed up with. -> {"category": "account"}
Message: Do you have an office in Berlin? -> {"category": "other"}
'''
variants = {"zero_shot": BASE, "few_shot": BASE + SHOTS}

import pandas as pd
rows = []
for name, prefix in variants.items():
    for msg, gold in CASES:
        reply = ask(prefix + f"\nMessage: {msg}", temperature=0, max_tokens=100)
        try: pred, valid = extract_json(reply)["category"], True
        except Exception: pred, valid = None, False
        rows.append(dict(variant=name, message=msg, gold=gold, pred=pred, valid_json=valid, correct=pred == gold))
df = pd.DataFrame(rows)
report = df.groupby("variant")[["valid_json", "correct"]].mean().round(2)
display(report)
display(df[~df.correct])

### Testing report (fill in after running)
| Variant | Valid JSON | Accuracy | Notable failures |
|---|---|---|---|
| zero_shot | | | |
| few_shot | | | |

**Hypothesis for next iteration:** _e.g., add an example of a noisy/vague message and re-test._

## 7. The AI Productivity Toolkit
The reusable version of everything above (templates, workflows, tests, evaluation harness) lives in the GitHub repo. Replace the URL with yours.

In [ ]:
# !git clone https://github.com/<your-username>/ai-productivity-toolkit.git
# %cd ai-productivity-toolkit
# !pip -q install -r requirements.txt
# !python -m pytest -q
# from toolkit import LLM, workflows
# llm = LLM()
# print(workflows.summarize(llm, doc, audience="a new hire"))
# print(workflows.resume_bullets(llm, experience="Built a Flask API used by 3 teams.",
#       job_description="Backend engineer, Python, APIs", target_role="Backend Engineer"))

## 8. Key takeaways
- Structure beats cleverness: role + context + task + constraints + format.
- Few-shot examples fix format and edge-case ambiguity.
- Validate structured outputs; retry/repair once, then fail loudly.
- Measure prompts on labelled cases instead of judging by one output.
- Verify AI-written code, facts and resume claims before using them.